# 01. Build patient_index (psychosis-risk cohort + non-psychosis controls)

Produces `output/patient_index.parquet` using the patient-level cohort saved by
`PsychosisRiskCohort.ipynb` and a randomly sampled non-psychosis control group.

The downstream intermediate file names are intentionally unchanged, so notebooks
02-05 can be run in the same order without changing their input paths.

Schema of output:

| column | meaning |
|---|---|
| `row_id` | sequential row id, used downstream to align X/Y matrices |
| `person_id` | OMOP person identifier |
| `index_datetime` | for cases: psychosis-risk index date; for controls: a borrowed date from the case index distribution |
| `birth_datetime` | from `person.birth_datetime` |
| `AGE_AT_INDEX` | years between birth and index |
| `group` | `'psychosis_risk'` or `'non_psychosis'` |

Design choices:

- **Case cohort comes from `PsychosisRiskCohort.ipynb`.** The notebook looks for `outputs/psychosis_risk_cohort.parquet` first, then `output/psychosis_risk_cohort.parquet`.
- **Controls are sampled to match the case count by default.** Set `N_CONTROLS` manually if a different control size is needed.
- **Controls borrow index dates from cases.** This keeps the calendar-time distribution aligned with the case cohort.
- **Controls are age-restricted to the observed case age range at the borrowed index date.**
- **Strict control exclusion.** Persons in the psychosis-risk cohort, and persons ever coded with F20-F29, are excluded from the control pool.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

DATA_PATH = '/home/jupyter/2835794-data'
COHORT_PATH_CANDIDATES = [
    Path('outputs/psychosis_risk_cohort.parquet'),
    Path('output/psychosis_risk_cohort.parquet'),
    Path('psychosis_risk_cohort.parquet'),
]
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

RNG_SEED = 42
N_CONTROLS = None      # None = match the number of psychosis-risk cases
LOOKBACK_DAYS = 365    # recorded here for traceability; notebook 02 uses index_datetime as the cutoff

con = duckdb.connect()
print('DuckDB version:', duckdb.__version__)


## 1. Load psychosis-risk cohort as the case half

In [ ]:
cohort_path = next((p for p in COHORT_PATH_CANDIDATES if p.exists()), None)
if cohort_path is None:
    raise FileNotFoundError(
        'Could not find psychosis-risk cohort parquet. Run PsychosisRiskCohort.ipynb first. '        f'Checked: {[str(p) for p in COHORT_PATH_CANDIDATES]}'
    )

cohort = pd.read_parquet(cohort_path)
print('Psychosis-risk cohort:', cohort.shape, 'from', cohort_path)
print('Columns:', list(cohort.columns))

# Normalize expected column names from PsychosisRiskCohort.ipynb.
rename_map = {}
if 'index_date' in cohort.columns:
    rename_map['index_date'] = 'index_datetime'
elif 'INDEX_DATE' in cohort.columns:
    rename_map['INDEX_DATE'] = 'index_datetime'
elif 'index_datetime' not in cohort.columns:
    raise ValueError('Cohort must contain index_date, INDEX_DATE, or index_datetime.')

if 'age_at_index' in cohort.columns:
    rename_map['age_at_index'] = 'AGE_AT_INDEX'
elif 'AGE_AT_INDEX' not in cohort.columns:
    raise ValueError('Cohort must contain age_at_index or AGE_AT_INDEX.')

cases = cohort.rename(columns=rename_map).copy()
cases['index_datetime'] = pd.to_datetime(cases['index_datetime'])
cases['AGE_AT_INDEX'] = pd.to_numeric(cases['AGE_AT_INDEX'], errors='coerce')

# Attach birth_datetime from OMOP person table, since the risk-cohort export may not carry it.
case_birth = con.sql(f'''
SELECT person_id, CAST(birth_datetime AS TIMESTAMP) AS birth_datetime
FROM read_parquet('{DATA_PATH}/person/*.parquet')
WHERE person_id IN (SELECT person_id FROM cases)
''').to_df()
cases = cases.merge(case_birth, on='person_id', how='left')

cases = cases[['person_id', 'index_datetime', 'birth_datetime', 'AGE_AT_INDEX']].drop_duplicates('person_id')
cases['group'] = 'psychosis_risk'

if cases['birth_datetime'].isna().any():
    print('WARNING: missing birth_datetime for cases:', int(cases['birth_datetime'].isna().sum()))
if cases['AGE_AT_INDEX'].isna().any():
    cases['AGE_AT_INDEX'] = ((cases['index_datetime'] - cases['birth_datetime']).dt.days / 365.25)

CASE_MIN_AGE = int(np.floor(cases['AGE_AT_INDEX'].min()))
CASE_MAX_AGE = int(np.ceil(cases['AGE_AT_INDEX'].max()))
N_CONTROLS = len(cases) if N_CONTROLS is None else int(N_CONTROLS)

print('Cases:', len(cases))
print(f'Case age range used for controls: {CASE_MIN_AGE}-{CASE_MAX_AGE}')
print('Requested controls:', N_CONTROLS)
cases.head()


## 2. Identify psychosis-coded persons to exclude from the control pool

Strict definition: any person ever coded with ICD-10 F20-F29 is removed from the control pool. Persons already in the case cohort are also excluded.

In [ ]:
# Excluded codes for non-psychosis controls:
#   ICD-10 F20-F29: schizophrenia, schizotypal, delusional, and other non-mood psychotic disorders.

EXCLUSION_PATTERNS_SQL = '''
       regexp_matches(upper(trim(cond.condition_source_value)), '^F2[0-9]')
'''

psychosis_code_excluded = con.sql(f'''
SELECT DISTINCT person_id
FROM read_parquet('{DATA_PATH}/condition_occurrence/*.parquet') cond
WHERE cond.condition_source_value IS NOT NULL
  AND {EXCLUSION_PATTERNS_SQL}
''').to_df()

case_persons = cases[['person_id']].drop_duplicates()
excluded_persons = (
    pd.concat([psychosis_code_excluded, case_persons], ignore_index=True)
    .drop_duplicates()
    .reset_index(drop=True)
)
con.register('excluded_persons', excluded_persons)

print(f'Persons with F20-F29 codes: {len(psychosis_code_excluded):,}')
print(f'Total persons excluded from control pool including cases: {len(excluded_persons):,}')


## 3. Build the control candidate pool

A control candidate is any person who:
- appears in `person.parquet`
- is NOT in the excluded set above
- has at least one `condition_occurrence` record, indicating real EHR contact

In [ ]:
candidate_pool = con.sql(f'''
WITH all_persons AS (
    SELECT DISTINCT person_id, birth_datetime
    FROM read_parquet('{DATA_PATH}/person/*.parquet')
),
persons_with_any_condition AS (
    SELECT DISTINCT person_id
    FROM read_parquet('{DATA_PATH}/condition_occurrence/*.parquet')
)
SELECT p.person_id, p.birth_datetime
FROM all_persons p
WHERE p.person_id IN (SELECT person_id FROM persons_with_any_condition)
  AND p.person_id NOT IN (SELECT person_id FROM excluded_persons)
''').to_df()

print(f'Control candidate pool size: {len(candidate_pool):,}')


## 4. Sample controls with borrowed case index dates

- Each candidate is pre-assigned a borrowed `index_datetime` from a random case.
- Only candidates with `AGE_AT_INDEX` within the observed case age range are eligible.
- Sample `N_CONTROLS` without replacement from the eligible pool.

In [ ]:
rng = np.random.default_rng(RNG_SEED)

candidate_pool = candidate_pool.copy()
candidate_pool['birth_datetime'] = pd.to_datetime(candidate_pool['birth_datetime'])

# Step 1: pre-assign every candidate a borrowed index_datetime from the case distribution.
# The final control persons are unique, but borrowed dates may repeat across controls.
random_dates = (
    cases['index_datetime']
    .sample(n=len(candidate_pool), replace=True, random_state=RNG_SEED)
    .reset_index(drop=True)
)
candidate_pool = candidate_pool.reset_index(drop=True)
candidate_pool['index_datetime'] = random_dates
candidate_pool['AGE_AT_INDEX'] = (
    (candidate_pool['index_datetime'] - candidate_pool['birth_datetime']).dt.days / 365.25
)

# Step 2: keep candidates within the case cohort's observed age range at borrowed index.
eligible = candidate_pool[
    (candidate_pool['AGE_AT_INDEX'] >= CASE_MIN_AGE) &
    (candidate_pool['AGE_AT_INDEX'] <= CASE_MAX_AGE)
].reset_index(drop=True)
print(f'Candidates with age {CASE_MIN_AGE}-{CASE_MAX_AGE} at borrowed index: {len(eligible):,}')

if len(eligible) < N_CONTROLS:
    raise ValueError(
        f'Eligible pool ({len(eligible)}) smaller than N_CONTROLS ({N_CONTROLS}).'
    )

# Step 3: sample N_CONTROLS without replacement.
sampled_idx = rng.choice(len(eligible), size=N_CONTROLS, replace=False)
controls = eligible.iloc[sampled_idx].reset_index(drop=True).copy()
controls['group'] = 'non_psychosis'
controls = controls[['person_id', 'index_datetime', 'birth_datetime', 'AGE_AT_INDEX', 'group']]

print(f'Final controls sampled: {len(controls):,}')
print()
print('Case vs control age summary:')
print(pd.DataFrame({
    'case': cases['AGE_AT_INDEX'].describe(),
    'control': controls['AGE_AT_INDEX'].describe(),
}))


## 5. Combine cases + controls into one `patient_index` table

In [ ]:
patient_index = pd.concat(
    [cases[['person_id', 'index_datetime', 'birth_datetime', 'AGE_AT_INDEX', 'group']],
     controls[['person_id', 'index_datetime', 'birth_datetime', 'AGE_AT_INDEX', 'group']]],
    ignore_index=True
)
patient_index['row_id'] = np.arange(1, len(patient_index) + 1)
patient_index = patient_index[['row_id', 'person_id', 'index_datetime',
                                'birth_datetime', 'AGE_AT_INDEX', 'group']]

print('Combined patient_index shape:', patient_index.shape)
print()
print('Group counts:')
print(patient_index['group'].value_counts())
print()
print('Age distribution by group:')
print(patient_index.groupby('group')['AGE_AT_INDEX'].describe())
print()
print('Index date range by group:')
print(patient_index.groupby('group')['index_datetime'].agg(['min', 'max']))


## 6. Save

In [ ]:
out_path = OUTPUT_DIR / 'patient_index.parquet'
patient_index.to_parquet(out_path, index=False)
print(f'Saved: {out_path}  ({len(patient_index):,} rows)')

# Keep the original intermediate filename so downstream/checking scripts do not need changes.
excluded_persons.to_parquet(OUTPUT_DIR / 'control_excluded_persons.parquet', index=False)
print(f'Saved exclusion list: {len(excluded_persons):,} persons')
